# Tribal Fire Response and Capacity Gap Analysis

**Series:** Tribal Fire Science & Indigenous Data Sovereignty  
**Author:** Lilly Jones, PhD  
**Last Updated:** 2025  
**Data Sources:** Census TIGER AIANNH, OpenStreetMap (fire stations via osmnx)

## Overview
This notebook analyzes fire response capacity and coverage gaps on Tribal lands by mapping
nearby fire stations and calculating straight-line distance buffers to estimate coverage
zones. Fire station locations are downloaded from OpenStreetMap.

Coverage is modeled as straight-line (Euclidean) distance buffers, a standard approach
for rural fire coverage analysis where road networks are sparse or unavailable.
Straight-line buffers represent a **best-case** coverage scenario; actual road-network
travel times will be longer, particularly on Tribal lands where road infrastructure
is often limited. For real-life scenarios, download road networks from OpenStreetMap.

## Research Questions
- Which Tribal lands fall outside straight-line coverage of a fire station?
- Which areas should be prioritized for new stations or mutual aid agreements?
- How does coverage vary across the ten highest fire-frequency Tribal lands?

## Outputs
- Per-Tribe coverage maps with buffer zones
- Coverage gap analysis per Tribal land unit
- Priority ranking for response capacity improvements

## Data Sovereignty Note
> Tribal boundary data comes from Census TIGER AIANNH federal statistical boundaries
> that may not reflect Tribal Nations' own definitions of territory.
> Tribal fire program staffing data is not available from a public API; enter known
> values in `TRIBAL_FIRE_PROGRAMS` in Section 1.
> All geospatial data is real; no synthetic data is used.

In [ ]:
# Imports
import sys
from pathlib import Path

REPO_ROOT = Path().resolve().parent
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import warnings
from datetime import datetime

import contextily as ctx
import geopandas as gpd
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import osmnx as ox
import pandas as pd
import seaborn as sns
from shapely.validation import make_valid

from src.data import constants, loaders, validators
from src.data.constants import PRIMARY_TRIBES
from src.geo import utils as geo_utils
from src.indigenous.sovereignty import generate_citations, print_data_acknowledgment
from src.viz import charts, styles

styles.apply_mpl_style()
%matplotlib inline

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning, module="geopandas")

print(f"Repo root : {REPO_ROOT}")
print(f"Output dir: {constants.OUTPUTS_DIR}")
print(f"Analysis run: {datetime.now().strftime('%Y-%m-%d %H:%M')}")

In [ ]:
print_data_acknowledgment(source_keys=["census_aiannh"])

## Configuration

In [ ]:
ANALYSIS_CONFIG = {
    "buffer_radii_km":      [10, 20, 30],
    "max_coverage_km":      30,
    "coverage_target_pct":  90,
    "station_query_buffer": 0.5,  # degrees, per-tribe bbox expansion for OSM query
}

TRIBES_OF_INTEREST = PRIMARY_TRIBES

# Tribal fire program staffing is not available from a public API.
# Update with values obtained directly from Tribal fire managers or
# BIA Division of Fire Management for real-world scenarios.
TRIBAL_FIRE_PROGRAMS = {
    "Choctaw":                               {"has_program": None, "staff_count": None},
    "Osage":                                 {"has_program": None, "staff_count": None},
    "Creek":                                 {"has_program": None, "staff_count": None},
    "San Carlos":                            {"has_program": True,  "staff_count": None},
    "Fort Apache":                           {"has_program": True,  "staff_count": 12},
    "Cherokee":                              {"has_program": None, "staff_count": None},
    "Kiowa-Comanche-Apache-Fort Sill Apache":{"has_program": None, "staff_count": None},
    "Chickasaw":                             {"has_program": None, "staff_count": None},
    "Colville":                              {"has_program": True,  "staff_count": 20},
    "Warm Springs":                          {"has_program": True,  "staff_count": 12},
}

print(f"Buffer radii   : {ANALYSIS_CONFIG['buffer_radii_km']} km")
print(f"Coverage target: {ANALYSIS_CONFIG['coverage_target_pct']}%")
print(f"Tribal Nations : {len(TRIBES_OF_INTEREST)}")

## Load Data

In [ ]:
# Tribal land boundaries 
all_tribal = loaders.load_census_aian()
all_tribal = validators.validate_geodataframe(
    all_tribal, "census_aiannh", required_columns=["geometry", "NAME"]
)

tribal_lands = all_tribal[all_tribal["NAME"].isin(TRIBES_OF_INTEREST)].copy()
tribal_lands = tribal_lands.dissolve(by="NAME", as_index=False).reset_index(drop=True)
tribal_lands["geometry"] = tribal_lands.geometry.apply(
    lambda g: make_valid(g) if g is not None else g
)
CONUS = geo_utils.bbox_geodataframe((-127, 24, -65, 50)).geometry.iloc[0]
tribal_lands = tribal_lands[
    tribal_lands.geometry.notnull() &
    tribal_lands.geometry.is_valid &
    tribal_lands.intersects(CONUS)
].copy().reset_index(drop=True)

tribal_lands["has_fire_program"] = tribal_lands["NAME"].map(
    lambda n: TRIBAL_FIRE_PROGRAMS.get(n, {}).get("has_program")
)
tribal_lands["fire_staff_count"] = tribal_lands["NAME"].map(
    lambda n: TRIBAL_FIRE_PROGRAMS.get(n, {}).get("staff_count")
)

print(f"Tribal land units: {len(tribal_lands)}")
print(tribal_lands[["NAME", "has_fire_program", "fire_staff_count"]].to_string(index=False))

In [ ]:
# Fire stations via OpenStreetMap
# Query one small bbox per Tribal land to avoid large-area timeouts.

ox.settings.log_console = False
ox.settings.use_cache   = True

print("Fetching fire stations from OpenStreetMap...")
buf = ANALYSIS_CONFIG["station_query_buffer"]
all_raw = []

for _, tribe in tribal_lands.iterrows():
    tb = tribe.geometry.bounds
    try:
        raw = ox.features_from_bbox(
            bbox=(tb[0]-buf, tb[1]-buf, tb[2]+buf, tb[3]+buf),
            tags={"amenity": "fire_station"},
        )
        all_raw.append(raw)
        print(f"  {tribe['NAME']}: {len(raw)} stations")
    except Exception as e:
        print(f"  {tribe['NAME']}: none found or query failed — {e}")

if all_raw:
    stations_raw  = pd.concat(all_raw).drop_duplicates()
    fire_stations = stations_raw.copy()
    fire_stations["geometry"] = fire_stations.geometry.centroid
    fire_stations = gpd.GeoDataFrame(
        fire_stations, geometry="geometry", crs=constants.CRS_GEOGRAPHIC
    ).reset_index(drop=True)
    fire_stations = fire_stations[
        fire_stations.geometry.notnull() & ~fire_stations.geometry.is_empty
    ].reset_index(drop=True)
    print(f"\nTotal unique fire stations: {len(fire_stations):,}")
else:
    raise RuntimeError("No fire stations retrieved from OSM.")

## Coverage Analysis

In [ ]:
# Build union coverage polygon at max radius 
stations_proj    = fire_stations.to_crs("EPSG:5070")
max_km           = ANALYSIS_CONFIG["max_coverage_km"]
coverage_union_proj = (
    stations_proj.buffer(max_km * 1000).union_all()
)

# Per-Tribe coverage
tribal_proj = tribal_lands.to_crs("EPSG:5070").copy()
results = []
for _, tribe in tribal_proj.iterrows():
    total_area   = tribe.geometry.area
    covered      = tribe.geometry.intersection(coverage_union_proj)
    covered_area = covered.area if not covered.is_empty else 0
    gap_geom     = tribe.geometry.difference(coverage_union_proj)
    coverage_pct = (covered_area / total_area * 100) if total_area > 0 else 0
    results.append({
        "NAME":             tribe["NAME"],
        "has_fire_program": tribe.get("has_fire_program"),
        "fire_staff_count": tribe.get("fire_staff_count"),
        "coverage_pct":     round(coverage_pct, 1),
        "gap_pct":          round(100 - coverage_pct, 1),
        "geometry":         tribe.geometry,
        "gap_geometry":     gap_geom if not gap_geom.is_empty else None,
    })

coverage_gdf = (
    gpd.GeoDataFrame(results, crs="EPSG:5070")
    .to_crs(constants.CRS_GEOGRAPHIC)
)

print(f"COVERAGE ANALYSIS ({max_km} km straight-line buffer — best case)")
print("=" * 65)
print(
    coverage_gdf[["NAME", "coverage_pct", "gap_pct", "has_fire_program"]]
    .sort_values("coverage_pct")
    .to_string(index=False)
)
print(f"\nMean coverage: {coverage_gdf['coverage_pct'].mean():.1f}%")

## Priority Ranking

In [ ]:
def prioritize_gaps(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Score each Tribal land by urgency of response capacity improvement."""
    df = gdf.copy()
    df["priority_score"] = df["gap_pct"] * 0.4
    df["priority_score"] += df["has_fire_program"].map(
        {False: 30, True: 0, None: 15}
    ).fillna(15)
    max_staff = df["fire_staff_count"].max()
    if pd.notna(max_staff) and max_staff > 0:
        df["priority_score"] += df["fire_staff_count"].apply(
            lambda x: (1 - x / max_staff) * 30 if pd.notna(x) else 15
        )
    else:
        df["priority_score"] += 15
    df["priority_category"] = pd.cut(
        df["priority_score"], bins=[0, 35, 65, 100],
        labels=["Low", "Medium", "High"],
    )
    return df.sort_values("priority_score", ascending=False)


priority_df = prioritize_gaps(coverage_gdf)
print(
    priority_df[[
        "NAME", "coverage_pct", "has_fire_program",
        "fire_staff_count", "priority_score", "priority_category",
    ]].to_string(index=False)
)

## Visualizations

In [ ]:
# Per-Tribe coverage maps 
iso_colors = {10: "#27AE60", 20: "#F39C12", 30: "#E67E22"}
n_tribes   = len(tribal_lands)
ncols, nrows = 2, (n_tribes + 1) // 2

fig, axes = plt.subplots(nrows, ncols, figsize=(14, nrows * 5))
axes = axes.flatten()

for idx, (_, tribe_row) in enumerate(tribal_lands.iterrows()):
    ax   = axes[idx]
    name = tribe_row["NAME"]
    tb   = tribe_row.geometry.bounds
    buf_deg = 0.6
    clip_box = geo_utils.bbox_geodataframe(
        (tb[0]-buf_deg, tb[1]-buf_deg, tb[2]+buf_deg, tb[3]+buf_deg)
    ).geometry.iloc[0]

    local_stations = fire_stations[fire_stations.within(clip_box)]
    tribe_3857     = gpd.GeoDataFrame(
        geometry=[tribe_row.geometry], crs=constants.CRS_GEOGRAPHIC
    ).to_crs(3857)

    if not local_stations.empty:
        local_proj = local_stations.to_crs("EPSG:5070")
        for r_km, color in sorted(iso_colors.items(), reverse=True):
            buf_union = local_proj.buffer(r_km * 1000).union_all()
            gpd.GeoDataFrame(geometry=[buf_union], crs="EPSG:5070").to_crs(3857).plot(
                ax=ax, color=color, alpha=0.18, edgecolor="none"
            )
        local_stations.to_crs(3857).plot(
            ax=ax, color=styles.EMBER_RED, marker="*",
            markersize=80, edgecolor="black", linewidth=0.4, zorder=5,
        )

    tribe_3857.plot(
        ax=ax, facecolor=styles.EARTH_BROWN,
        edgecolor=styles.CHARCOAL, alpha=0.35, linewidth=1.2,
    )
    try:
        ctx.add_basemap(ax, source=ctx.providers.CartoDB.Positron)
    except Exception:
        pass

    cov_row = coverage_gdf[coverage_gdf["NAME"] == name]
    cov_pct = cov_row["coverage_pct"].iloc[0] if not cov_row.empty else 0
    ax.set_title(f"{name}\nCoverage: {cov_pct:.0f}%", fontsize=9, fontweight="bold")
    ax.set_axis_off()

for ax in axes[n_tribes:]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpatches.Patch(facecolor=c, alpha=0.4, label=f"{r} km buffer")
        for r, c in iso_colors.items()
    ] + [
        mpatches.Patch(facecolor=styles.EARTH_BROWN, alpha=0.4, label="Tribal land"),
        plt.Line2D([0],[0], marker="*", color="w",
                   markerfacecolor=styles.EMBER_RED, markersize=10, label="Fire station"),
    ],
    loc="lower center", ncol=5, fontsize=9, bbox_to_anchor=(0.5, 0.01),
)
plt.suptitle(
    f"Fire Station Coverage Straight-Line Buffers\n"
    f"(Best-case estimate; actual road travel times will be longer)",
    fontsize=13, fontweight="bold",
)
plt.tight_layout(rect=[0, 0.05, 1, 1])
charts.save_figure(fig, "outputs/figures/tribal_fire_coverage_map.png")
plt.show()

In [ ]:
# Summary bar charts
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

cov = coverage_gdf.sort_values("coverage_pct")
bar_colors = [
    styles.SAGE_GREEN  if v >= ANALYSIS_CONFIG["coverage_target_pct"]
    else styles.FIRE_ORANGE if v >= 50
    else styles.EMBER_RED
    for v in cov["coverage_pct"]
]
ax1.barh(cov["NAME"], cov["coverage_pct"], color=bar_colors, alpha=0.85)
ax1.axvline(
    ANALYSIS_CONFIG["coverage_target_pct"],
    color=styles.CHARCOAL, linestyle="--", linewidth=1.5,
    label=f"{ANALYSIS_CONFIG['coverage_target_pct']}% target",
)
ax1.set_xlabel("Coverage (%)")
ax1.set_title(
    f"Coverage per Tribal Land ({max_km} km straight-line)",
    fontsize=11, fontweight="bold",
)
ax1.legend(fontsize=9)
ax1.set_xlim(0, 105)
sns.despine(ax=ax1)

p_colors = {"High": styles.EMBER_RED, "Medium": styles.FIRE_ORANGE, "Low": styles.SAGE_GREEN}
ax2.barh(
    priority_df["NAME"],
    priority_df["priority_score"],
    color=[p_colors.get(str(c), styles.SMOKE_GRAY) for c in priority_df["priority_category"]],
    alpha=0.85,
)
ax2.set_xlabel("Priority Score")
ax2.set_title("Response Capacity Priority Score", fontsize=11, fontweight="bold")
ax2.legend(
    handles=[mpatches.Patch(color=v, label=k) for k, v in p_colors.items()],
    fontsize=9,
)
sns.despine(ax=ax2)

plt.suptitle("Tribal Fire Response Capacity Analysis", fontsize=13, fontweight="bold")
plt.tight_layout()
charts.save_figure(fig, "outputs/figures/tribal_fire_capacity_charts.png")
plt.show()

## Exports

In [ ]:
coverage_gdf[
    ["NAME", "has_fire_program", "fire_staff_count", "coverage_pct", "gap_pct", "geometry"]
].to_file(constants.OUTPUTS_DIR / "tribal_fire_coverage.geojson", driver="GeoJSON")
print("Exported to outputs/tribal_fire_coverage.geojson")

fire_stations.to_file(
    constants.OUTPUTS_DIR / "fire_stations_osm.geojson", driver="GeoJSON"
)
print("Exported to outputs/fire_stations_osm.geojson")

priority_df[
    ["NAME", "coverage_pct", "gap_pct", "has_fire_program",
     "fire_staff_count", "priority_score", "priority_category"]
].to_csv(constants.OUTPUTS_DIR / "tribal_fire_capacity_priority.csv", index=False)
print("Exported to outputs/tribal_fire_capacity_priority.csv")

## Summary & Findings

*(Fill in after running.)*

Questions to address:
- Which Tribal lands have the largest coverage gaps within 30 km?
- For Tribes with unknown fire program data (`None`), what would direct
  engagement with Tribal fire managers reveal? Update `TRIBAL_FIRE_PROGRAMS`
  as information becomes available.
- What are the limitations? Straight-line buffers overestimate coverage;
  actual road travel on Tribal lands, which often have limited infrastructure,
  will be substantially longer. This is a best-case estimate, not a realistic
  response time estimate.
- OSM may not include all Tribal-operated fire stations, particularly
  smaller volunteer departments. Ground-truthing with BIA Division of
  Fire Management records is recommended.

---
## References

In [ ]:
print(generate_citations(["census_aiannh"]))